In [1]:
import sys
print(sys.executable)

c:\Users\umaka\anaconda3\python.exe


In [2]:
%pip install mlxtend

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

c:\Users\umaka\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
df = pd.read_csv("../data/smartplate_features.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset Shape: (21900, 45)


,Date,Day_of_Week,Food_Item,Food_Category,Meal_Type,Quantity_Prepared,Quantity_Sold,Quantity_Wasted,Actual_Demand,Number_of_Students,...,Category_Avg_Waste_Percentage,Meal_Avg_Waste_Percentage,Year,Month,Month_Name,Weekday_Number,Weekday_Name,Is_Weekend,Demand_Per_Student,Demand_Based_Preparation_Buffer
0,2025-01-01,Wednesday,Buttermilk,Beverage,Beverages,41.07,39.09,1.98,39.09,171.55,...,6.680912,6.680912,2025,1,January,2,Wednesday,0,0.227864,1.98
1,2025-01-01,Wednesday,Coffee,Beverage,Beverages,41.47,40.10,1.37,40.10,171.55,...,6.680912,6.680912,2025,1,January,2,Wednesday,0,0.233751,1.37
2,2025-01-01,Wednesday,Lemon Juice,Beverage,Beverages,40.15,37.21,2.94,37.21,171.55,...,6.680912,6.680912,2025,1,January,2,Wednesday,0,0.216905,2.94
3,2025-01-01,Wednesday,Tea,Beverage,Beverages,77.84,57.53,20.31,57.53,171.55,...,6.680912,6.680912,2025,1,January,2,Wednesday,0,0.335354,20.31
4,2025-01-01,Wednesday,Aloo Paratha,Main Course,Breakfast,40.98,39.88,1.10,39.88,171.55,...,6.620998,6.666646,2025,1,January,2,Wednesday,0,0.232469,1.10


In [5]:
required_columns = [
    "Date",
    "Meal_Type",
    "Food_Item",
    "High_Waste_Flag"
]

print("Required columns:")
print(df[required_columns].head())

print("\nHigh Waste Flag Distribution:")
print(df["High_Waste_Flag"].value_counts())

Required columns:
         Date  Meal_Type     Food_Item  High_Waste_Flag
0  2025-01-01  Beverages    Buttermilk                0
1  2025-01-01  Beverages        Coffee                0
2  2025-01-01  Beverages   Lemon Juice                1
3  2025-01-01  Beverages           Tea                1
4  2025-01-01  Breakfast  Aloo Paratha                0

High Waste Flag Distribution:
High_Waste_Flag
0    10950
1    10950
Name: count, dtype: int64


In [6]:
high_waste_df = df[df["High_Waste_Flag"] == 1].copy()

print("Total high-waste records:", len(high_waste_df))

high_waste_df[
    ["Date", "Meal_Type", "Food_Item", "High_Waste_Flag"]
].head()

Total high-waste records: 10950


,Date,Meal_Type,Food_Item,High_Waste_Flag
2,2025-01-01,Beverages,Lemon Juice,1
3,2025-01-01,Beverages,Tea,1
8,2025-01-01,Breakfast,Poha,1
11,2025-01-01,Dinner,Chicken Biryani,1
13,2025-01-01,Dinner,Paneer Masala,1


In [7]:
high_waste_df["Transaction_ID"] = (
    high_waste_df["Date"].astype(str)
    + "_"
    + high_waste_df["Meal_Type"].astype(str)
)

high_waste_df[
    ["Transaction_ID", "Food_Item"]
].head(10)

,Transaction_ID,Food_Item
2,2025-01-01_Beverages,Lemon Juice
3,2025-01-01_Beverages,Tea
8,2025-01-01_Breakfast,Poha
11,2025-01-01_Dinner,Chicken Biryani
13,2025-01-01_Dinner,Paneer Masala
14,2025-01-01_Dinner,Veg Biryani
15,2025-01-01_Dinner,Veg Thali
16,2025-01-01_Lunch,Chapati
17,2025-01-01_Lunch,Chicken Biryani
18,2025-01-01_Lunch,Chole Rice


In [8]:
transactions = (
    high_waste_df
    .groupby("Transaction_ID")["Food_Item"]
    .apply(list)
    .tolist()
)

print("Total Transactions:", len(transactions))

print("\nFirst 5 Transactions:")
for transaction in transactions[:5]:
    print(transaction)

Total Transactions: 3574

First 5 Transactions:
['Lemon Juice', 'Tea']
['Poha']
['Chicken Biryani', 'Paneer Masala', 'Veg Biryani', 'Veg Thali']
['Chapati', 'Chicken Biryani', 'Chole Rice', 'Dal Rice', 'Paneer Rice', 'Rajma Rice', 'Veg Thali']
['Pav Bhaji', 'Samosa', 'Vada Pav']


In [9]:
transactions = [
    transaction
    for transaction in transactions
    if len(transaction) >= 2
]

print("Transactions with 2 or more items:", len(transactions))

Transactions with 2 or more items: 3110


In [10]:
te = TransactionEncoder()

transaction_array = te.fit(transactions).transform(transactions)

transaction_df = pd.DataFrame(
    transaction_array,
    columns=te.columns_
)

print("Transaction Dataset Shape:", transaction_df.shape)

transaction_df.head()

Transaction Dataset Shape: (3110, 25)


,Aloo Paratha,Buttermilk,Chapati,Chicken Biryani,Chole Rice,Coffee,Dal Rice,Dal Tadka,Idli Sambar,Lemon Juice,...,Pav Bhaji,Poha,Rajma Rice,Samosa,Sandwich,Tea,Upma,Vada Pav,Veg Biryani,Veg Thali
0,False,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,True,False,False,False,False
1,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True
2,False,False,True,True,True,False,True,False,False,False,...,False,False,True,False,False,False,False,False,False,True
3,False,False,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,False,True,False,False
4,False,False,False,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False


In [11]:
frequent_itemsets = apriori(
    transaction_df,
    min_support=0.02,
    use_colnames=True
)

frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)

print("Total Frequent Itemsets:", len(frequent_itemsets))

frequent_itemsets.sort_values(
    by="support",
    ascending=False
).head(10)

Total Frequent Itemsets: 230


,support,itemsets,length
24,0.235370,frozenset({Veg Thali}),1
2,0.235048,frozenset({Chapati}),1
6,0.227974,frozenset({Dal Rice}),1
3,0.227653,frozenset({Chicken Biryani}),1
23,0.222186,frozenset({Veg Biryani}),1
63,0.121543,"frozenset({Veg Thali, Dal Rice})",2
35,0.119936,"frozenset({Dal Rice, Chapati})",2
49,0.119614,"frozenset({Veg Thali, Chicken Biryani})",2
41,0.119293,"frozenset({Veg Thali, Chapati})",2
33,0.118328,"frozenset({Chicken Biryani, Chapati})",2


In [12]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.50
)

print("Total Association Rules:", len(rules))

rules.head()

Total Association Rules: 271


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({Aloo Paratha}),frozenset({Masala Dosa}),0.111897,0.116399,0.060772,0.543103,4.665889,1.0,0.047747,1.933920,0.884671,0.362764,0.482915,0.532601
1,frozenset({Masala Dosa}),frozenset({Aloo Paratha}),0.116399,0.111897,0.060772,0.522099,4.665889,1.0,0.047747,1.858342,0.889178,0.362764,0.461886,0.532601
2,frozenset({Misal Pav}),frozenset({Aloo Paratha}),0.114469,0.111897,0.062058,0.542135,4.844941,1.0,0.049249,1.939660,0.896185,0.377691,0.484446,0.548366
3,frozenset({Aloo Paratha}),frozenset({Misal Pav}),0.111897,0.114469,0.062058,0.554598,4.844941,1.0,0.049249,1.988159,0.893589,0.377691,0.497022,0.548366
4,frozenset({Poha}),frozenset({Aloo Paratha}),0.114148,0.111897,0.057235,0.501408,4.480978,1.0,0.044462,1.781223,0.876935,0.339048,0.438588,0.506451


In [13]:
meaningful_rules = rules[
    (rules["confidence"] >= 0.50) &
    (rules["lift"] > 1)
].copy()

meaningful_rules = meaningful_rules.sort_values(
    by=["lift", "confidence"],
    ascending=False
)

print("Meaningful Rules Found:", len(meaningful_rules))

meaningful_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].head(10)

Meaningful Rules Found: 271


,antecedents,consequents,support,confidence,lift
36,frozenset({Lemon Juice}),frozenset({Coffee}),0.060129,0.593651,5.861124
37,frozenset({Coffee}),frozenset({Lemon Juice}),0.060129,0.593651,5.861124
12,frozenset({Buttermilk}),frozenset({Tea}),0.054662,0.536278,5.634538
13,frozenset({Tea}),frozenset({Buttermilk}),0.054662,0.574324,5.634538
39,frozenset({Coffee}),frozenset({Tea}),0.052733,0.520635,5.470184
38,frozenset({Tea}),frozenset({Coffee}),0.052733,0.554054,5.470184
8,frozenset({Buttermilk}),frozenset({Coffee}),0.056270,0.552050,5.450403
9,frozenset({Coffee}),frozenset({Buttermilk}),0.056270,0.555556,5.450403
56,frozenset({Tea}),frozenset({Lemon Juice}),0.052090,0.547297,5.403475
55,frozenset({Lemon Juice}),frozenset({Tea}),0.052090,0.514286,5.403475


In [14]:
meaningful_rules["antecedents"] = meaningful_rules[
    "antecedents"
].apply(lambda x: ", ".join(list(x)))

meaningful_rules["consequents"] = meaningful_rules[
    "consequents"
].apply(lambda x: ", ".join(list(x)))

final_rules = meaningful_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].copy()

final_rules.head(10)

,antecedents,consequents,support,confidence,lift
36,Lemon Juice,Coffee,0.060129,0.593651,5.861124
37,Coffee,Lemon Juice,0.060129,0.593651,5.861124
12,Buttermilk,Tea,0.054662,0.536278,5.634538
13,Tea,Buttermilk,0.054662,0.574324,5.634538
39,Coffee,Tea,0.052733,0.520635,5.470184
38,Tea,Coffee,0.052733,0.554054,5.470184
8,Buttermilk,Coffee,0.056270,0.552050,5.450403
9,Coffee,Buttermilk,0.056270,0.555556,5.450403
56,Tea,Lemon Juice,0.052090,0.547297,5.403475
55,Lemon Juice,Tea,0.052090,0.514286,5.403475


In [15]:
final_rules.to_csv(
    "../output/apriori_association_rules.csv",
    index=False
)

print("Association Rules saved successfully!")

Association Rules saved successfully!
